In [1]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

load_dotenv()
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [6]:
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# Step 1: Create documents (you would fetch these from DB)
documents = [
    Document(page_content="Saraswati Public Library is a well-known public library located in Lucknow, Uttar Pradesh. It was established in the year 1998 to promote reading and education among the local community."),
    
    Document(page_content="The library operates under the supervision of the State Library Department and is registered with the government under the registration number LIB-UP-1998-0456."),
    
    Document(page_content="The Library is situated at 123, Hazratganj Road, in a prime area of Lucknow, making it easily accessible to residents and visitors."),
    
    Document(page_content="The library opens every day at 9:00 AM and closes at 7:00 PM, remaining open from Monday to Saturday and closed on Sundays."),
    
    Document(page_content="The facility houses a vast collection of over 75,000 books across multiple genres including literature, science, and technology."),
    
    Document(page_content="The library provides access to a digital library, including e-books and online journals."),
    Document(page_content="The library remains closed on public holiday's like Holi, Diwali."),
]

# Step 2: Initialize embeddings
embeddings = OpenAIEmbeddings()

# Step 3: Create in-memory vector store (FAISS)
vector_store = FAISS.from_documents(documents, embeddings)

# Step 4: User query
query = "What is your Library address?"

# Step 5: Perform similarity search
results = vector_store.similarity_search(query, k=3)

# Step 6: Print results
for i, doc in enumerate(results, 1):
    print(f"\nResult {i}:")
    print(doc.page_content)


Result 1:
The library operates under the supervision of the State Library Department and is registered with the government under the registration number LIB-UP-1998-0456.

Result 2:
The Library is situated at 123, Hazratganj Road, in a prime area of Lucknow, making it easily accessible to residents and visitors.

Result 3:
The facility houses a vast collection of over 75,000 books across multiple genres including literature, science, and technology.


In [7]:
context = "\n\n".join([doc.page_content for doc in results])

In [58]:
prompt = f"""
You are a helpful assistant for a library system based in India.

Use the following context to answer the question.
If the answer is not in the context, say "I don't know".

Context:
{context}

Question:
{query}

Answer:
"""

In [59]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o", temperature=0)

response = llm.invoke(prompt)

print(response.content)

The Library is situated at 123, Hazratganj Road, in a prime area of Lucknow.


In [9]:
class AgentState(TypedDict):
    user_input: str
    intent: str
    response: str
    confirmation: str  # "yes" | "no" | None

In [10]:
def ask_confirmation(state: AgentState):
    state["response"] = "Do you want to register this complaint? (yes/no)"
    return state

In [11]:
from langgraph.graph import END

def wait_for_confirmation(state: AgentState):
    # Pause graph execution here
    return state  # LangGraph will wait for next input

In [12]:
def route_after_confirmation(state: AgentState):
    answer = state["user_input"].lower()

    if "yes" in answer:
        return "save_complaint"
    else:
        return "cancel"

In [13]:
def save_complaint(state: AgentState):
    # db.insert_complaint(...)
    state["response"] = "Your complaint has been registered."
    return state

In [15]:
def cancel_complaint(state: AgentState):
    state["response"] = "Okay, complaint not registered."
    return state

In [17]:
builder = StateGraph(AgentState)
builder.add_node("ask_confirmation", ask_confirmation)
builder.add_node("save_complaint", save_complaint)
builder.add_node("cancel", cancel_complaint)

# From classifier
builder.add_edge("complaint", "ask_confirmation")

# After asking → WAIT for next user input
builder.add_conditional_edges(
    "ask_confirmation",
    route_after_confirmation,
    {
        "save_complaint": "save_complaint",
        "cancel": "cancel",
    }
)

In [ ]:
state = {
    "user_input": "The seats are dirty",
}
result = graph.invoke(state)

print(result["response"])

In [25]:
import uuid
from typing import Optional
from typing_extensions import TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.constants import START
from langgraph.graph import StateGraph
from langgraph.types import interrupt, Command

class State(TypedDict):
    """The graph state."""

    foo: str
    human_value: Optional[str]
    """Human value will be updated using an interrupt."""

def node(state: State):
    answer = interrupt(
        # This value will be sent to the client
        # as part of the interrupt information.
        "what is your age?"
    )
    print(f"> Received an input from the interrupt: {answer}")
    return {"human_value": answer}

builder = StateGraph(State)
builder.add_node("node", node)
builder.add_edge(START, "node")

# A checkpointer must be enabled for interrupts to work!
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

config = {
    "configurable": {
        "thread_id": uuid.uuid4(),
    }
}

for chunk in graph.stream({"foo": "abc"}, config):
    print(chunk)



{'__interrupt__': (Interrupt(value='what is your age?', id='2003f56bf7a7756b3dbe5c5bcb0a6fcf'),)}


In [23]:
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

command = Command(resume="My age is 39!!!")

for chunk in graph.stream(Command(resume="some input from a human!!!"), config):
    print(chunk)

# > Received an input from the interrupt: some input from a human!!!
# > {'node': {'human_value': 'some input from a human!!!'}}

{'__interrupt__': (Interrupt(value='what is your age?', id='79ebb0bacaf2bc2d6a8662cd85cbd7ed'),)}


In [26]:
command = Command(resume="My age is 39!!!")

for chunk in graph.stream(command, config):
    print(chunk)

> Received an input from the interrupt: My age is 39!!!
{'node': {'human_value': 'My age is 39!!!'}}
